In [ ]:
!pip install ultralytics -q
import os, shutil, xml.etree.ElementTree as ET
from pathlib import Path
from ultralytics import YOLO
import yaml

In [ ]:
import os

base = "/kaggle/input/datasets/sreekaraditya/rdd2022-yolo-crackscan-v2"  # verify this path first
train_imgs = os.listdir(f"{base}/train/images")
val_imgs   = os.listdir(f"{base}/val/images")
test_imgs  = os.listdir(f"{base}/test/images")

print(f"Train: {len(train_imgs)} images")
print(f"Val:   {len(val_imgs)} images")
print(f"Test:  {len(test_imgs)} images")

In [ ]:
from collections import Counter

label_dir = f"{base}/train/labels"
class_counts = Counter()

for label_file in os.listdir(label_dir):
    with open(f"{label_dir}/{label_file}") as f:
        for line in f:
            class_id = int(line.strip().split()[0])
            class_counts[class_id] += 1

class_names = {0:"Longitudinal", 1:"Transverse", 2:"Alligator", 3:"Pothole"}
for cid, count in sorted(class_counts.items()):
    print(f"Class {cid} ({class_names[cid]}): {count} instances")

In [ ]:
# This tells YOLO to penalize missing a pothole more than missing a crack
# Weight = inverse frequency, normalized
# Longitudinal: 22083, Transverse: 10032, Alligator: 9025, Pothole: 5539

total = 22083 + 10032 + 9025 + 5539  # = 46679
weights = [
    total/22083,  # 0 Longitudinal → 2.11
    total/10032,  # 1 Transverse   → 4.65
    total/9025,   # 2 Alligator    → 5.17
    total/5539    # 3 Pothole      → 8.43  ← highest, model focuses here
]
print(weights)
# [2.11, 4.65, 5.17, 8.43]

In [ ]:
import yaml

# First find exact dataset path
for root, dirs, files in os.walk("/kaggle/input"):
    if "data.yaml" in files:
        print(os.path.join(root, "data.yaml"))
        break

In [ ]:
dataset_path = "/kaggle/input/datasets/sreekaraditya/rdd2022-yolo-crackscan-v2"  

new_yaml = {
    "path": dataset_path,
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc": 4,
    "names": ["Longitudinal", "Transverse", "Alligator", "Pothole"]
}

with open("/kaggle/working/data.yaml", "w") as f:
    yaml.dump(new_yaml, f)

print("yaml written to /kaggle/working/data.yaml")

In [ ]:
import yaml
with open("/kaggle/working/data.yaml") as f:
    print(yaml.safe_load(f))

In [ ]:
#confirm the path actually exists:
import os
path = "/kaggle/input/datasets/sreekaraditya/rdd2022-yolo-crackscan-v2"
print(os.path.exists(path))                          # should print True
print(len(os.listdir(f"{path}/train/images")))       # should print ~32000

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data     = "/kaggle/working/data.yaml",
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    lr0      = 0.01,
    patience = 10,
    augment  = True,
    cls      = 2.0,
    device   = 0,
    project  = "/kaggle/working/runs",
    name     = "rdd_v1",
    exist_ok = True
)

In [ ]:
from ultralytics import YOLO
import os
import glob

model = YOLO("/kaggle/working/runs/rdd_v1/weights/best.pt")

# Grab ONLY 10 images
val_images = glob.glob(
    "/kaggle/input/datasets/sreekaraditya/rdd2022-yolo-crackscan-v2/val/images/*.jpg"
)[:10]

print(f"Testing on {len(val_images)} images")

results = model.predict(
    source  = val_images,
    conf    = 0.20,
    save    = True,
    project = "/kaggle/working/inference_test",
    name    = "sample_10",
)

for i, r in enumerate(results):
    if len(r.boxes) > 0:
        classes_detected = [model.names[int(c)] for c in r.boxes.cls]
        confs = [round(float(c), 2) for c in r.boxes.conf]
        print(f"Image {i}: {list(zip(classes_detected, confs))}")
    else:
        print(f"Image {i}: nothing detected")

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/rdd_v1/weights/best.pt")

# Export to ONNX for backend use
model.export(format="onnx", imgsz=640)
print("ONNX exported")

# Clean inference utility function
# This is what your friend integrates into FastAPI
def detect_damage(image_path: str, conf: float = 0.28):
    """
    Input:  path to any road image
    Output: list of detections with class, confidence, bounding box
    """
    model = YOLO("/kaggle/working/runs/rdd_v1/weights/best.pt")
    results = model.predict(image_path, conf=conf, verbose=False)[0]
    
    detections = []
    for box in results.boxes:
        detections.append({
            "class":      results.names[int(box.cls)],
            "confidence": round(float(box.conf), 3),
            "bbox":       box.xyxy[0].tolist(),  # [x1, y1, x2, y2] pixels
            "severity":   "high" if float(box.conf) > 0.5 else "medium"
        })
    
    return {
        "total_detections": len(detections),
        "has_pothole": any(d["class"] == "Pothole" for d in detections),
        "detections": detections
    }

# Test it
test = detect_damage(val_images[0])
print(test)